In [1]:
# Importing Libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [3]:
# Loading Dataset
df = pd.read_csv("BloodPressureData.csv")
print(df)

    BP  Age  Score    Sex
0  124   22     65    Man
1  117   33     34  Woman
2  127   35     34    Man
3  132   35     35    Man
4  117   42     54  Woman
5  122   52     60  Woman
6  119   62     60  Woman
7  134   65     42    Man


In [4]:
# Converting Sex into numerical value
df["Sex"] = df["Sex"].map({"Man": 1, "Woman": 0})

In [5]:
# Splitting the dataset with X and y
X = df[["Age", "Score", "Sex"]]
y = df["BP"]

features = ["Age", "Score", "Sex"]

n = len(y)

In [6]:
# Storing all possible subsets
subsets = []

# Null model
subsets.append([])

# One-feature models
for i in range(3):
    subsets.append([features[i]])

# Two-feature models
for i in range(3):
    for j in range(i + 1, 3):
        subsets.append([
            features[i],
            features[j]
        ])

# Three-feature model
subsets.append([
    features[0],
    features[1],
    features[2]
])

In [8]:
# Best Subset Selection
results = []
for subset in subsets:
    if len(subset) == 0:
        y_pred = np.full(n, y.mean())

    else:
        X_subset = X[subset]
        model = LinearRegression()
        model.fit(X_subset, y)

        y_pred = model.predict(X_subset)

    rss = np.sum((y - y_pred) ** 2)
    tss = np.sum((y - y.mean()) ** 2)
    r2 = 1 - (rss / tss)
    
    k = len(subset)
    if k == 0:
        adjusted_r2 = np.nan
    else:
        adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - k - 1))

    results.append([", ".join(subset) if subset else "None", k, rss, r2, adjusted_r2])

In [9]:
# Display all models
results_df = pd.DataFrame(
    results,
    columns=[
        "Features",
        "Number of Features",
        "RSS",
        "R²",
        "Adjusted R²"
    ]
)

print("\nAll Possible Models:")
print(results_df)


All Possible Models:
          Features  Number of Features         RSS        R²  Adjusted R²
0             None                   0  300.000000  0.000000          NaN
1              Age                   1  294.223754  0.019254    -0.144203
2            Score                   1  254.747107  0.150843     0.009317
3              Sex                   1   79.500000  0.735000     0.690833
4       Age, Score                   2  243.245354  0.189182    -0.135145
5         Age, Sex                   2   32.006985  0.893310     0.850634
6       Score, Sex                   2   75.474122  0.748420     0.647787
7  Age, Score, Sex                   3   26.307279  0.912309     0.846541


In [10]:
# Finding best model for each subset size
best_models = []
for k in range(4):
    models = results_df[results_df["Number of Features"] == k]
    # Minimum RSS
    best = models.loc[models["RSS"].idxmin()]
    best_models.append(best)

best_models_df = pd.DataFrame(best_models)

print("\nBest Model for Each Subset Size:")
print(best_models_df)


Best Model for Each Subset Size:
          Features  Number of Features         RSS        R²  Adjusted R²
0             None                   0  300.000000  0.000000          NaN
3              Sex                   1   79.500000  0.735000     0.690833
5         Age, Sex                   2   32.006985  0.893310     0.850634
7  Age, Score, Sex                   3   26.307279  0.912309     0.846541


In [12]:
# Selecting final model using Adjusted R²
final_model = best_models_df.loc[best_models_df["Adjusted R²"].idxmax()]

print("\nFinal Best Subset:")
print(final_model)


Final Best Subset:
Features               Age, Sex
Number of Features            2
RSS                   32.006985
R²                      0.89331
Adjusted R²            0.850634
Name: 5, dtype: object
